# Ecuaci?n de calor 1D: ejercicio guiado (con checks)

**Objetivo pedag?gico:** construir un solver FTCS desde cero, verificar estabilidad y cuantificar error.

Estudiamos:

$$
\frac{\partial u}{\partial t}=\alpha\frac{\partial^2u}{\partial x^2},
\quad x\in(0,1),\; t\in(0,T)
$$

con fronteras homog?neas y condici?n inicial:

$$u(0,t)=u(1,t)=0,\qquad u(x,0)=\sin(\pi x).$$

## 1) Contexto f?sico

Esta ecuaci?n modela difusi?n t?rmica: los picos de temperatura se suavizan con el tiempo.

- Si hay gradientes muy pronunciados, el calor fluye r?pido.
- A largo plazo, con fronteras fijas en cero, la temperatura tiende a cero en todo el dominio.

## 2) Discretizaci?n FTCS

Con malla uniforme:

$$x_i=i\,\Delta x,\quad t^n=n\,\Delta t$$

y aproximaciones finitas:

$$
\frac{u_i^{n+1}-u_i^n}{\Delta t}
=
\alpha\frac{u_{i+1}^n-2u_i^n+u_{i-1}^n}{\Delta x^2}.
$$

Despejando:

$$
u_i^{n+1}=u_i^n+r(u_{i+1}^n-2u_i^n+u_{i-1}^n),
\qquad r=\alpha\frac{\Delta t}{\Delta x^2}.
$$

**Condici?n de estabilidad del esquema expl?cito:** $r\le 1/2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
# Par?metros base
alpha = 1.0
L = 1.0
T = 0.12
N = 80

dx = L / (N + 1)
dt = 0.45 * dx**2 / alpha
r = alpha * dt / dx**2
print(f"dx={dx:.6f}, dt={dt:.8f}, r={r:.3f}")

## 3) Funciones de apoyo (exacta y malla)

In [ ]:
def exact_solution(x, t, alpha=1.0):
    return np.exp(-alpha*np.pi**2*t) * np.sin(np.pi*x)

def initial_condition(x):
    return np.sin(np.pi*x)

## 4) TODO A: construye la malla interior y n?mero de pasos temporales

In [ ]:
def build_grids(L, T, N, dt):
    """
    Regresa:
      x : nodos interiores (N nodos)
      nt: entero de pasos de tiempo para llegar a T (o superarlo ligeramente)
    """
    # TODO
    # dx = ...
    # x = ...
    # nt = ...
    raise NotImplementedError("Completa build_grids")

## 5) TODO B: implementa un paso FTCS con fronteras homog?neas

In [ ]:
def ftcs_step(u, r):
    """
    u: vector interior en tiempo n
    r: numero de difusi?n
    """
    # TODO
    # unew = np.empty_like(u)
    # extremo izquierdo usa frontera u(0,t)=0
    # extremo derecho usa frontera u(1,t)=0
    # interior con slicing
    raise NotImplementedError("Completa ftcs_step")

## 6) TODO C: solver completo + almacenamiento de historial para animaci?n

In [ ]:
def solve_heat_ftcs(alpha=1.0, L=1.0, T=0.12, N=80, dt=None, store_every=6):
    if dt is None:
        dx_local = L / (N + 1)
        dt = 0.45 * dx_local**2 / alpha

    x, nt = build_grids(L, T, N, dt)
    dx_local = L / (N + 1)
    r_local = alpha * dt / dx_local**2

    if r_local > 0.5:
        raise ValueError(f"Esquema inestable: r={r_local:.4f} > 0.5")

    u = initial_condition(x)
    U_hist = [u.copy()]
    t_hist = [0.0]

    # TODO: ciclo temporal usando ftcs_step
    # for n in range(1, nt+1):
    #     ...

    # retorna ?ltimo estado y tambi?n historial
    # return x, u, t_final, r_local, np.array(t_hist), np.array(U_hist)
    raise NotImplementedError("Completa solve_heat_ftcs")

## 7) Checks b?sicos autom?ticos

In [ ]:
x, u_num, t_final, r, t_hist, U_hist = solve_heat_ftcs(alpha=alpha, L=L, T=T, N=N, dt=dt)

assert isinstance(x, np.ndarray) and isinstance(u_num, np.ndarray)
assert x.ndim == 1 and u_num.ndim == 1
assert len(x) == N and len(u_num) == N
assert r <= 0.5 + 1e-12
assert np.all(np.isfinite(u_num))
assert len(t_hist) == len(U_hist)

u_ex = exact_solution(x, t_final, alpha=alpha)
err_l2 = np.sqrt(np.mean((u_num - u_ex)**2))
err_inf = np.max(np.abs(u_num - u_ex))

print(f"Checks OK")
print(f"t_final={t_final:.6f}, r={r:.4f}")
print(f"Error L2   = {err_l2:.3e}")
print(f"Error Linf = {err_inf:.3e}")

## 8) Visualizaci?n: perfil final

In [ ]:
u_ex = exact_solution(x, t_final, alpha=alpha)

plt.figure(figsize=(8,4))
plt.plot(x, u_num, 'o-', ms=3, label='Num?rica FTCS')
plt.plot(x, u_ex, '--', lw=2, label='Exacta')
plt.xlabel('x')
plt.ylabel('u(x,T)')
plt.title('Comparaci?n en tiempo final')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 9) Visualizaci?n: animaci?n en notebook

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
line_num, = ax.plot([], [], lw=2, label='FTCS')
line_ex, = ax.plot([], [], '--', lw=1.8, label='Exacta')
txt = ax.text(0.02, 0.92, '', transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.set_title('Evoluci?n de temperatura')
ax.grid(alpha=0.25)
ax.legend()

def init():
    line_num.set_data([], [])
    line_ex.set_data([], [])
    txt.set_text('')
    return line_num, line_ex, txt

def update(k):
    tk = t_hist[k]
    line_num.set_data(x, U_hist[k])
    line_ex.set_data(x, exact_solution(x, tk, alpha=alpha))
    txt.set_text(f't={tk:.4f}')
    return line_num, line_ex, txt

ani = FuncAnimation(fig, update, frames=len(t_hist), init_func=init, interval=80, blit=True)
plt.close(fig)
ani

## 10) Retos sugeridos

1. Repite con $u(x,0)=\sin(2\pi x)$ y compara la rapidez de decaimiento.
2. Fija $N$ y aumenta $r$ hasta violar estabilidad. Describe qu? observas.
3. Haz una tabla de errores para varios $N$ manteniendo $r$ constante.